# 🏦 Data Foundation | Notebook 01
## Geração de Bases Sintéticas — Mercado Financeiro

---

**Série:** #DataFoundation  
**Autor:** Léo Touguinha | Especialista em Dados | Mercado Financeiro  
**Repositório:** github.com/LeoTouguinha/data-foundation

---

## Contexto

Antes de qualquer análise, pipeline ou modelo, existe uma decisão que define a qualidade de tudo que vem depois: **como os dados estão estruturados e como se relacionam entre si.**

Este notebook gera 6 bases sintéticas que simulam o ambiente real de dados de uma corretora no mercado financeiro — com relacionamentos, distribuições realistas e problemas propositais de qualidade para demonstração de pipeline completo.

> **Os parâmetros deste notebook são baseados em operação real:**  
> 100.000 cotações/mês, funil de 3 estágios, mix de modalidades 10/70/20.  
> Os dados gerados são sintéticos. Os números de negócio são reais.

---

## Bases Geradas

| # | Base | Registros | Descrição |
|---|------|-----------|----------|
| 1 | `clientes` | 1.000 | Cadastro base — chave mestra |
| 2 | `transacoes` | 10.000 | Movimentações financeiras |
| 3 | `status_clientes` | 1.000 | Ativo / Inativo (70/30) |
| 4 | `classificacao_clientes` | 1.000 | Segmentação A/B/C por valor |
| 5 | `faixa_salarial` | 1.000 | Renda com distribuição 50/30/20 |
| 6 | `produtos_contratados` | ~2.500 | Produtos por cliente (N:N) |
| RAW | `transacoes_raw` | ~10.300 | Versão com erros de qualidade |

---

## Diagrama de Relacionamento

```
                    ┌─────────────────┐
                    │    CLIENTES     │  ← Tabela Mestra
                    │  (id_cliente)   │
                    └────────┬────────┘
           ┌─────────────────┼─────────────────────┐
           │                 │                     │
    ┌──────▼──────┐  ┌───────▼──────┐  ┌──────────▼───────┐
    │  TRANSAÇÕES │  │    STATUS    │  │  CLASSIFICAÇÃO   │
    │  (10.000)   │  │   (1.000)    │  │    (1.000)       │
    └─────────────┘  └──────────────┘  └──────────────────┘
                     ┌──────────────┐  ┌──────────────────┐
                     │FAIXA SALARIAL│  │    PRODUTOS      │
                     │  (1.000)     │  │  CONTRATADOS     │
                     └──────────────┘  │   (~2.500)       │
                                       └──────────────────┘
```

## 0. Configuração

In [3]:
import sys
sys.path.append("../src")

import random
import uuid
import csv
import copy
import os
import pandas as pd
from datetime import datetime, timedelta
from utils import (
    SEED, NOMES_BRASILEIROS, CANAIS_PESOS,
    gerar_cpf_fake, gerar_id_transacao,
    data_aleatoria, separador, formatar_reais
)

random.seed(SEED)

OUTPUT_DIR = "../data/raw"
os.makedirs(OUTPUT_DIR, exist_ok=True)

def salvar_e_visualizar(nome: str, cabecalho: list, linhas: list, preview: int = 5):
    caminho = f"{OUTPUT_DIR}/{nome}"
    with open(caminho, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=cabecalho)
        writer.writeheader()
        writer.writerows(linhas)
    df = pd.DataFrame(linhas)
    print(f"\n✅ {nome} — {len(linhas):,} registros")
    print(f"   Colunas: {list(df.columns)}")
    display(df.head(preview))
    return df

print(f"✅ Configuração ok | Seed: {SEED} | Saída: {OUTPUT_DIR}")

✅ Configuração ok | Seed: 42 | Saída: ../data/raw


## 1. Base de Clientes — 1.000 registros

**Tabela mestra.** Todos os `id_cliente` das demais bases vêm daqui.  
Em produção, esta seria a camada **Gold / Trusted** do cadastro unificado.

## ⚠️ Classificação de Dados — LGPD

| Campo | Classificação | Controle em produção |
|---|---|---|
| `cpf` | **Restrito** — dado pessoal identificável (LGPD Art. 5°) | Mascaramento na Bronze, pseudonimização na Silver |
| `nome_cliente` | **Restrito** — dado pessoal | Mascaramento na Bronze |
| `id_cliente` | **Confidencial** — chave de negócio | RBAC + log de acesso |
| `valor_reais` | **Confidencial** — dado financeiro | Criptografia em repouso |
| `canal` | **Interno** | Autenticação básica |
| `data_transacao` | **Interno** | Autenticação básica |

> **Nota:** Neste projeto os dados são **sintéticos**. Em produção, `cpf` e `nome_cliente` seriam mascarados na ingestão Bronze e nunca expostos em CSV. A camada Silver exporia apenas `id_cliente` (pseudônimo).
>
> **Base legal (LGPD):** processamento para execução de contrato (Art. 7°, V) com retenção conforme prazo regulatório BACEN (5 anos).


In [4]:
clientes     = []
ids_clientes = []

for i in range(1, 1_001):
    id_cliente = f"CLI{str(i).zfill(5)}"
    ids_clientes.append(id_cliente)
    clientes.append({
        "id"           : i,
        "id_cliente"   : id_cliente,
        "nome_cliente" : random.choice(NOMES_BRASILEIROS),
        "cpf"          : gerar_cpf_fake(),
        "data_cadastro": data_aleatoria("2018-01-01", "2023-12-31"),
    })

df_clientes = salvar_e_visualizar(
    "clientes.csv",
    ["id", "id_cliente", "nome_cliente", "cpf", "data_cadastro"],
    clientes,
)


✅ clientes.csv — 1,000 registros
   Colunas: ['id', 'id_cliente', 'nome_cliente', 'cpf', 'data_cadastro']


,id,id_cliente,nome_cliente,cpf,data_cadastro
0,1,CLI00001,Henrique Alves,043.321.819-64,2018-05-11
1,2,CLI00002,Bruno Souza,133.890.838-63,2020-06-21
2,3,CLI00003,Camila Dias,940.265.423-53,2019-02-23
3,4,CLI00004,Felipe Rocha,615.594.078-25,2022-03-31
4,5,CLI00005,Felipe Rocha,849.593.103-47,2018-11-23


## 2. Transações Financeiras — 10.000 registros

Simula extrato de movimentações com canal, valor e data.  
Campo `canal` com pesos realistas de adoção digital brasileira:  
PIX (40%) > App (30%) > Internet Banking (20%) > Agência (7%) > Totem (3%)

In [5]:
FAIXAS_VALOR = {
    "debito" : (10.0,   5_000.0),
    "credito": (50.0,  15_000.0),
}

transacoes = []
for i in range(1, 10_001):
    tipo = random.choice(["debito", "credito"])
    v_min, v_max = FAIXAS_VALOR[tipo]
    transacoes.append({
        "id"            : i,
        "id_transacao"  : gerar_id_transacao(),
        "id_cliente"    : random.choice(ids_clientes),
        "tipo_transacao": tipo,
        "valor_reais"   : round(random.uniform(v_min, v_max), 2),
        "canal"         : random.choice(CANAIS_PESOS),
        "data_transacao": data_aleatoria("2023-01-01", "2024-12-31"),
    })

df_transacoes = salvar_e_visualizar(
    "transacoes.csv",
    ["id", "id_transacao", "id_cliente", "tipo_transacao",
     "valor_reais", "canal", "data_transacao"],
    transacoes,
)


✅ transacoes.csv — 10,000 registros
   Colunas: ['id', 'id_transacao', 'id_cliente', 'tipo_transacao', 'valor_reais', 'canal', 'data_transacao']


,id,id_transacao,id_cliente,tipo_transacao,valor_reais,canal,data_transacao
0,1,TXND7D06B9221,CLI00630,debito,4535.18,pix,2024-08-17
1,2,TXNF2811E3CAF,CLI00131,credito,9332.30,internet_banking,2023-10-28
2,3,TXNB107A3E694,CLI00774,debito,4227.09,pix,2024-02-17
3,4,TXN273D4E6F50,CLI00474,debito,4039.33,app_mobile,2024-04-16
4,5,TXN369C608D60,CLI00087,debito,282.92,internet_banking,2023-04-20


## 3. Status dos Clientes — 1.000 registros

Distribuição: **70% Ativos / 30% Inativos** — ratio típico em carteiras de banco digital.  
Base para análise de churn e campanhas de reativação.

In [6]:
STATUS_POOL = ["ativo"] * 70 + ["inativo"] * 30

status_clientes = []
for i, id_cliente in enumerate(ids_clientes, start=1):
    status_clientes.append({
        "id"              : i,
        "id_cliente"      : id_cliente,
        "status"          : random.choice(STATUS_POOL),
        "data_atualizacao": data_aleatoria("2023-01-01", "2024-12-31"),
    })

df_status = salvar_e_visualizar(
    "status_clientes.csv",
    ["id", "id_cliente", "status", "data_atualizacao"],
    status_clientes,
)

print("\n📊 Distribuição de Status:")
print(df_status["status"].value_counts(normalize=True).mul(100).round(1).to_string())


✅ status_clientes.csv — 1,000 registros
   Colunas: ['id', 'id_cliente', 'status', 'data_atualizacao']


,id,id_cliente,status,data_atualizacao
0,1,CLI00001,ativo,2023-09-27
1,2,CLI00002,inativo,2024-07-18
2,3,CLI00003,ativo,2024-07-15
3,4,CLI00004,ativo,2023-10-06
4,5,CLI00005,ativo,2024-08-31



📊 Distribuição de Status:
status
ativo      70.2
inativo    29.8


## 4. Classificação dos Clientes — 1.000 registros

Segmentação por valor para o negócio (não por renda):  
- **A** (20%) — Alta rentabilidade  
- **B** (30%) — Média rentabilidade  
- **C** (50%) — Baixa rentabilidade  

> Regra de Pareto: os 20% da classe A geralmente representam 80% da receita.

In [7]:
CLASSIF_POOL = ["A"] * 20 + ["B"] * 30 + ["C"] * 50

classificacao_clientes = []
for i, id_cliente in enumerate(ids_clientes, start=1):
    classificacao_clientes.append({
        "id"                : i,
        "id_cliente"        : id_cliente,
        "classificacao"     : random.choice(CLASSIF_POOL),
        "data_classificacao": data_aleatoria("2023-01-01", "2024-12-31"),
    })

df_classif = salvar_e_visualizar(
    "classificacao_clientes.csv",
    ["id", "id_cliente", "classificacao", "data_classificacao"],
    classificacao_clientes,
)

print("\n📊 Distribuição de Classificação:")
print(df_classif["classificacao"].value_counts(normalize=True).mul(100).round(1).to_string())


✅ classificacao_clientes.csv — 1,000 registros
   Colunas: ['id', 'id_cliente', 'classificacao', 'data_classificacao']


,id,id_cliente,classificacao,data_classificacao
0,1,CLI00001,C,2024-02-27
1,2,CLI00002,B,2023-11-04
2,3,CLI00003,C,2023-01-17
3,4,CLI00004,C,2023-01-22
4,5,CLI00005,A,2024-08-27



📊 Distribuição de Classificação:
classificacao
C    51.5
B    29.2
A    19.3


## 5. Faixa Salarial — 1.000 registros

Distribuição controlada: **50% Classe A | 30% Classe B | 20% Classe C**

| Classe | Faixa | % da Base |
|--------|-------|-----------|
| A | R$ 0 – R$ 10.000 | 50% |
| B | R$ 10.001 – R$ 50.000 | 30% |
| C | R$ 50.001 – R$ 100.000 | 20% |

In [8]:
FAIXAS = [
    ("0 a 1.000",             0,       1_000, "A"),
    ("1.001 a 3.000",     1_001,       3_000, "A"),
    ("3.001 a 6.000",     3_001,       6_000, "A"),
    ("6.001 a 10.000",    6_001,      10_000, "A"),
    ("10.001 a 15.000",  10_001,      15_000, "B"),
    ("15.001 a 20.000",  15_001,      20_000, "B"),
    ("20.001 a 30.000",  20_001,      30_000, "B"),
    ("30.001 a 50.000",  30_001,      50_000, "B"),
    ("50.001 a 100.000", 50_001,     100_000, "C"),
]

FAIXAS_A = [f for f in FAIXAS if f[3] == "A"]
FAIXAS_B = [f for f in FAIXAS if f[3] == "B"]
FAIXAS_C = [f for f in FAIXAS if f[3] == "C"]

N_A, N_B = 500, 300
N_C = 1_000 - N_A - N_B

pool = ["A"] * N_A + ["B"] * N_B + ["C"] * N_C
random.shuffle(pool)

clientes_embaralhados = ids_clientes.copy()
random.shuffle(clientes_embaralhados)

def sortear_salario(faixas):
    f = random.choice(faixas)
    return f[0], round(random.uniform(f[1], f[2]), 2), f[3]

faixa_salarial = []
mapa_faixas = {"A": FAIXAS_A, "B": FAIXAS_B, "C": FAIXAS_C}
for i, classe in enumerate(pool, start=1):
    label, salario, classif = sortear_salario(mapa_faixas[classe])
    faixa_salarial.append({
        "id"            : i,
        "id_cliente"    : clientes_embaralhados[i - 1],
        "faixa_salarial": label,
        "salario"       : salario,
        "classificacao" : classif,
    })

df_salarial = salvar_e_visualizar(
    "faixa_salarial.csv",
    ["id", "id_cliente", "faixa_salarial", "salario", "classificacao"],
    faixa_salarial,
)

print("\n📊 Distribuição por Classificação Salarial:")
print(df_salarial["classificacao"].value_counts(normalize=True).mul(100).round(1).to_string())


✅ faixa_salarial.csv — 1,000 registros
   Colunas: ['id', 'id_cliente', 'faixa_salarial', 'salario', 'classificacao']


,id,id_cliente,faixa_salarial,salario,classificacao
0,1,CLI00011,1.001 a 3.000,1649.97,A
1,2,CLI00587,0 a 1.000,122.56,A
2,3,CLI00115,0 a 1.000,27.07,A
3,4,CLI00368,50.001 a 100.000,84681.63,C
4,5,CLI00646,3.001 a 6.000,4715.26,A



📊 Distribuição por Classificação Salarial:
classificacao
A    50.0
B    30.0
C    20.0


## 6. Produtos Contratados — ~2.500 registros

Relacionamento **N:N** — um cliente pode ter múltiplos produtos.  
Simula cross-sell típico de bancos e fintechs.  

> Clientes Classe A têm probabilidade maior de contratar mais produtos — regra de negócio explícita no código.

In [9]:
PRODUTOS = [
    "conta_corrente", "cartao_credito", "cartao_debito",
    "credito_pessoal", "investimento_renda_fixa",
    "investimento_renda_variavel", "seguro_vida",
    "consorcio", "previdencia_privada",
]

classif_map = {r["id_cliente"]: r["classificacao"] for r in classificacao_clientes}
PRODUTOS_POR_CLASSE = {"A": (3, 6), "B": (2, 4), "C": (1, 3)}

produtos_contratados = []
linha_id = 1
for id_cliente in ids_clientes:
    classe = classif_map.get(id_cliente, "C")
    min_p, max_p = PRODUTOS_POR_CLASSE[classe]
    n_prod = random.randint(min_p, max_p)
    escolhidos = random.sample(PRODUTOS, k=min(n_prod, len(PRODUTOS)))
    for produto in escolhidos:
        produtos_contratados.append({
            "id"              : linha_id,
            "id_cliente"      : id_cliente,
            "produto"         : produto,
            "data_contratacao": data_aleatoria("2019-01-01", "2024-12-31"),
            "status_produto"  : random.choice(["ativo"] * 85 + ["cancelado"] * 15),
        })
        linha_id += 1

df_produtos = salvar_e_visualizar(
    "produtos_contratados.csv",
    ["id", "id_cliente", "produto", "data_contratacao", "status_produto"],
    produtos_contratados,
)

print("\n📊 Top produtos por volume:")
print(df_produtos["produto"].value_counts().to_string())


✅ produtos_contratados.csv — 2,779 registros
   Colunas: ['id', 'id_cliente', 'produto', 'data_contratacao', 'status_produto']


,id,id_cliente,produto,data_contratacao,status_produto
0,1,CLI00001,credito_pessoal,2020-09-28,ativo
1,2,CLI00002,credito_pessoal,2023-09-27,cancelado
2,3,CLI00002,seguro_vida,2022-07-02,ativo
3,4,CLI00003,previdencia_privada,2020-08-28,cancelado
4,5,CLI00004,previdencia_privada,2019-12-25,ativo



📊 Top produtos por volume:
produto
cartao_debito                  327
investimento_renda_fixa        324
cartao_credito                 322
seguro_vida                    319
previdencia_privada            310
credito_pessoal                305
consorcio                      297
conta_corrente                 289
investimento_renda_variavel    286


## 7. Base RAW — Transações com Erros de Qualidade

Esta base simula como os dados **chegam na camada Bronze** antes de qualquer tratamento.

| Problema | % | Impacto |
|----------|---|---------|
| Nulos em campos críticos | ~5% | Análises incompletas |
| Duplicatas | ~3% | Dupla contagem de receita |
| `id_cliente` inválido | ~2% | Orphan records — quebra de FK |
| Valor negativo | ~2% | Anomalia contábil |
| Data futura | ~1% | Erro de sistema |

> No notebook 02, usamos PySpark para detectar e tratar todos esses problemas.

In [10]:
transacoes_raw = copy.deepcopy(transacoes)
n = len(transacoes_raw)

# 1. Nulos (~5%)
for idx in random.sample(range(n), k=int(n * 0.05)):
    transacoes_raw[idx][random.choice(["valor_reais", "canal", "tipo_transacao"])] = None

# 2. Duplicatas (~3%)
dups = [copy.deepcopy(transacoes_raw[idx]) for idx in random.sample(range(n), k=int(n * 0.03))]
for i, d in enumerate(dups): d["id"] = n + i + 1
transacoes_raw.extend(dups)

# 3. FK inválida (~2%)
for idx in random.sample(range(n), k=int(n * 0.02)):
    transacoes_raw[idx]["id_cliente"] = f"CLI_INVALIDO_{random.randint(9000,9999)}"

# 4. Valor negativo (~2%)
for idx in random.sample(range(n), k=int(n * 0.02)):
    if transacoes_raw[idx]["valor_reais"]:
        transacoes_raw[idx]["valor_reais"] = -abs(transacoes_raw[idx]["valor_reais"])

# 5. Data futura (~1%)
for idx in random.sample(range(n), k=int(n * 0.01)):
    transacoes_raw[idx]["data_transacao"] = data_aleatoria("2025-06-01", "2027-12-31")

df_raw = salvar_e_visualizar(
    "transacoes_raw.csv",
    ["id", "id_transacao", "id_cliente", "tipo_transacao",
     "valor_reais", "canal", "data_transacao"],
    transacoes_raw, preview=8,
)

print(f"\n⚠️  Total com duplicatas: {len(transacoes_raw):,} (base original: 10.000)")


✅ transacoes_raw.csv — 10,300 registros
   Colunas: ['id', 'id_transacao', 'id_cliente', 'tipo_transacao', 'valor_reais', 'canal', 'data_transacao']


,id,id_transacao,id_cliente,tipo_transacao,valor_reais,canal,data_transacao
0,1,TXND7D06B9221,CLI00630,debito,4535.18,pix,2024-08-17
1,2,TXNF2811E3CAF,CLI_INVALIDO_9303,credito,9332.30,internet_banking,2023-10-28
2,3,TXNB107A3E694,CLI00774,debito,4227.09,pix,2024-02-17
3,4,TXN273D4E6F50,CLI00474,debito,4039.33,app_mobile,2024-04-16
4,5,TXN369C608D60,CLI00087,debito,282.92,internet_banking,2023-04-20
5,6,TXN67570F44F9,CLI00609,credito,1506.12,pix,2024-08-02
6,7,TXN074B298997,CLI00708,credito,5003.05,pix,2023-05-05
7,8,TXN7684E81B42,CLI00719,debito,1033.27,pix,2024-07-29



⚠️  Total com duplicatas: 10,300 (base original: 10.000)


## 8. Validação Final

In [11]:
print("=" * 55)
print("  RELATÓRIO DE GERAÇÃO — Data Foundation")
print("=" * 55)

bases = {
    "clientes"               : df_clientes,
    "transacoes"             : df_transacoes,
    "status_clientes"        : df_status,
    "classificacao_clientes" : df_classif,
    "faixa_salarial"         : df_salarial,
    "produtos_contratados"   : df_produtos,
    "transacoes_raw"         : df_raw,
}

for nome, df in bases.items():
    print(f"  ✔ {nome:<30} {len(df):>6,} linhas | {len(df.columns)} colunas")

print()
print("  🔗 Integridade referencial:")
ids_master = set(df_clientes["id_cliente"])
for nome, df in bases.items():
    if "id_cliente" in df.columns and nome != "transacoes_raw":
        orphans = df[~df["id_cliente"].isin(ids_master)]
        status = "✅ OK" if len(orphans) == 0 else f"⚠️  {len(orphans)} orphans"
        print(f"     {nome:<35} → {status}")

orphans_raw = df_raw[~df_raw["id_cliente"].isin(ids_master)]
print(f"     transacoes_raw                      → ⚠️  {len(orphans_raw)} orphans (PROPOSITAL)")
print()
print("  📁 Saída: data/raw/")
print("  🚀 Próximo: 02_qualidade_dados.ipynb")
print("=" * 55)

  RELATÓRIO DE GERAÇÃO — Data Foundation
  ✔ clientes                        1,000 linhas | 5 colunas
  ✔ transacoes                     10,000 linhas | 7 colunas
  ✔ status_clientes                 1,000 linhas | 4 colunas
  ✔ classificacao_clientes          1,000 linhas | 4 colunas
  ✔ faixa_salarial                  1,000 linhas | 5 colunas
  ✔ produtos_contratados            2,779 linhas | 5 colunas
  ✔ transacoes_raw                 10,300 linhas | 7 colunas

  🔗 Integridade referencial:
     clientes                            → ✅ OK
     transacoes                          → ✅ OK
     status_clientes                     → ✅ OK
     classificacao_clientes              → ✅ OK
     faixa_salarial                      → ✅ OK
     produtos_contratados                → ✅ OK
     transacoes_raw                      → ⚠️  200 orphans (PROPOSITAL)

  📁 Saída: data/raw/
  🚀 Próximo: 02_qualidade_dados.ipynb
